# exp116_hidden_like_anchor_score_readout_on_exp115 train

exp115 の hidden-like holdout split に、既存 anchor の OOF / train-side prediction を再学習なしで merge して採点する audit notebook。

## Contents

1. Setup and configuration
2. Input split and source inventory
3. Run hidden-like readout
4. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd

from hidden_like_anchor_score_readout_on_exp115 import first_existing_path, run_readout
from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_SOURCES_ENV = os.environ.get("EXPERIMENT_MAX_SOURCES")
MAX_SOURCES = int(MAX_SOURCES_ENV) if MAX_SOURCES_ENV else None

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", EXPERIMENT_NAME)
print("Route:", config["experiment"]["route"])
print("Parent:", config["lineage"]["parent"])
print("Mode:", config["readout"]["mode"])
print("Artifacts:", paths.artifacts_dir)
print("Debug:", DEBUG, "Max sources:", MAX_SOURCES)


## 2. Input split and source inventory


In [ ]:
exp115_config = config["readout"]["exp115"]
split_paths = {
    "fold_assignments": first_existing_path(paths, exp115_config["fold_assignments_path_candidates"]),
    "holdout_wells": first_existing_path(paths, exp115_config["holdout_wells_path_candidates"]),
    "well_metadata": first_existing_path(paths, exp115_config["well_metadata_path_candidates"]),
}
for name, path in split_paths.items():
    print(name, path, "exists=", path.exists() if path else False)

fold_assignments = pd.read_csv(split_paths["fold_assignments"])
holdout_wells = pd.read_csv(split_paths["holdout_wells"])
print("fold_assignments", fold_assignments.shape)
print("holdout_wells", holdout_wells.shape)
display(fold_assignments.head())
display(holdout_wells.groupby("variant").size().reset_index(name="wells"))

source_rows = []
for source in config["readout"]["prediction_sources"]:
    path = first_existing_path(paths, source["path_candidates"])
    source_rows.append(
        {
            "source": source["name"],
            "kind": source["kind"],
            "path": str(path) if path else None,
            "exists": bool(path),
        }
    )
display(pd.DataFrame(source_rows))


## 3. Run hidden-like readout


In [ ]:
summary = run_readout(allow_local=False, max_sources=MAX_SOURCES)
print("Loaded sources:", summary["loaded_sources"])
print("Missing sources:", summary["missing_sources"])
print(json.dumps(summary["best_overall_by_split"], indent=2)[:4000])


## 4. Metrics and artifacts


In [ ]:
artifact_paths = {key: Path(value) for key, value in summary["artifacts"].items()}
for name, path in artifact_paths.items():
    print(name, path, path.exists(), path.stat().st_size if path.exists() else None)

overall = pd.read_csv(artifact_paths["overall_metrics"])
bucket = pd.read_csv(artifact_paths["bucket_metrics"])
inventory = pd.read_csv(artifact_paths["source_inventory"])
display(inventory)
display(overall.sort_values(["split_variant", "rmse_tvt"]).head(20))
display(bucket.loc[bucket["bucket_family"].eq("eval_rank_bucket")].sort_values(["split_variant", "source", "model", "bucket"]).head(30))

metrics = json.loads(paths.metrics_path.read_text())
print(json.dumps(metrics, indent=2)[:4000])
